# Libraries

In [319]:
import pandas as pd
import seaborn as sns
from functools import reduce

# Resampling Libs
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

# Functions

## Resample

In [320]:
def fun_res(dfi):

    X = dfi.drop("Study_Status_Bin", axis = 1)
    y = dfi["Study_Status_Bin"]

    X_train_tts, X_test_tts, y_train_tts, y_test_tts = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

    res = RandomUnderSampler(sampling_strategy = 'auto', random_state = 42)
    X_train_res, y_train_res = res.fit_resample(X_train_tts, y_train_tts) 
    
    X_train_res = pd.DataFrame(X_train_res)

    return X_train_res, y_train_res



## Fun_Sparse

In [321]:
def fun_sparse(i, dfi, categ_cols):
    pivot_tables = []

    for col in categ_cols:
        if col not in dfi.columns:
            # αν δεν υπάρχει, φτιάξε dummy στήλη γεμάτη NaN/0
            dfi[col] = pd.Series([0]*len(dfi), index=dfi.index)

        pivot_table = pd.pivot_table(
            data = dfi,
            index = col,
            columns = "Study_Status_Bin",
            aggfunc = "size",
            fill_value = 0,
            observed = False
        ).reset_index()

        # Change Column/Element Names of Pivot
        pivot_table['Variables'] = col + ' = ' + pivot_table[col].astype(str)
        pivot_table.drop(columns=[col], inplace=True)

        # Reindex Column of value
        final_cols = ['Variables'] + [c for c in pivot_table.columns if c != 'Variables']
        pivot_table = pivot_table[final_cols]

        pivot_tables.append(pivot_table)

    # Merge all pivots
    pivot_merged = pd.concat(pivot_tables, ignore_index=True)
    pivot_merged = pivot_merged.rename(columns={0: f'0 - df{i}', 1: f'1 - df{i}'})
    
    return pivot_merged


## Fun_Zeros

In [322]:
def fun_zeros(pivot_merged, count, missing):

    num_cols = pivot_merged.select_dtypes(include='number')
    mask = num_cols < count
    if missing == True:
        mask |= num_cols.isna()
    sparse = pivot_merged[mask.any(axis=1)]
    return sparse

# Load Unmerged Data
- Data loaded haven't dropped first after dummies. All levels need checking
- Only on train data. 
- Test data are 'unseen' --> no sparsity check. 

In [323]:

df1 = pd.read_pickle(r".\df_dummies_unmerged\df1_dummies_unmerged.pkl")
df2 = pd.read_pickle(r".\df_dummies_unmerged\df2_dummies_unmerged.pkl")
df3 = pd.read_pickle(r".\df_dummies_unmerged\df3_dummies_unmerged.pkl")
df4 = pd.read_pickle(r".\df_dummies_unmerged\df4_dummies_unmerged.pkl")

## Resample

In [324]:
X_train1, y_train1 = fun_res(df1)
X_train2, y_train2 = fun_res(df2)
X_train3, y_train3 = fun_res(df3)
X_train4, y_train4 = fun_res(df4)

## Create X_y dfs

In [325]:
df1_train = pd.concat([X_train1, y_train1], axis=1)  # X_train y_train have same index
df2_train = pd.concat([X_train2, y_train2], axis=1)  # X_train y_train have same index
df3_train = pd.concat([X_train3, y_train3], axis=1)  # X_train y_train have same index
df4_train = pd.concat([X_train4, y_train4], axis=1)  # X_train y_train have same index

display(X_train1.shape)
display(X_train2.shape)
display(X_train3.shape)
display(X_train4.shape)

(5092, 169)

(8678, 170)

(3854, 168)

(3694, 169)

## Unique cols

In [326]:
dfs = [X_train1, X_train2, X_train3, X_train4]
all_unique_cols = set().union(*(df.columns for df in dfs))
print(f"{len(all_unique_cols)}")

# Columns missing from the dfs
for i, df in enumerate(dfs, start=1):
    missing = all_unique_cols - set(df.columns)
    extra = set(df.columns) - all_unique_cols
    print(f" X_train{i}: shape={df.shape}")
    print(f" Missing cols: {missing if missing else 'None'}")
    print(f" Extra cols:   {extra if extra else 'None'}\n")


172
 X_train1: shape=(5092, 169)
 Missing cols: {'Conditions_Detail_List_Chemical Actions and Uses', 'Conditions_Detail_List_Hemic and Immune Systems', 'Conditions_Detail_List_Fluids and Secretions'}
 Extra cols:   None

 X_train2: shape=(8678, 170)
 Missing cols: {'Conditions_Detail_List_Chemical Actions and Uses', 'Conditions_Detail_List_Fluids and Secretions'}
 Extra cols:   None

 X_train3: shape=(3854, 168)
 Missing cols: {'Conditions_Detail_List_Human Activities', 'Conditions_Detail_List_Hemic and Immune Systems', 'Conditions_Detail_List_Chemical Actions and Uses', 'Conditions_Detail_List_Information Science'}
 Extra cols:   None

 X_train4: shape=(3694, 169)
 Missing cols: {'Conditions_Detail_List_Human Activities', 'Conditions_Detail_List_Hemic and Immune Systems', 'Conditions_Detail_List_Fluids and Secretions'}
 Extra cols:   None



## Pivots

### Categ_Bin_Pivot

In [327]:
categ_cols = [col for col in all_unique_cols if '_Categ' in col or '_Bin' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, categ_cols)
pivot_merged2 = fun_sparse(2, df2_train, categ_cols)
pivot_merged3 = fun_sparse(3, df3_train, categ_cols)
pivot_merged4 = fun_sparse(4, df4_train, categ_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_categ = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_categ


In [328]:
sparse_categ_train = fun_zeros(pivot_merged_train_categ, 15, False) # No categorical cols were used. 
display(sparse_categ_train)

Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4


### List_Pivot

In [329]:
list_cols = [col for col in all_unique_cols if '_List' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, list_cols)
pivot_merged2 = fun_sparse(2, df2_train, list_cols)
pivot_merged3 = fun_sparse(3, df3_train, list_cols)
pivot_merged4 = fun_sparse(4, df4_train, list_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_list = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_list

In [342]:
sparse_list_train = fun_zeros(pivot_merged_train_list, 15, True)
display(sparse_list_train)


Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4
71,Conditions_Detail_List_Biological Phenomena = 1,2.0,1.0,3.0,5.0,0.0,2.0,3.0,4.0
75,Conditions_Detail_List_Cell Physiological Phen...,3.0,4.0,13.0,17.0,8.0,13.0,19.0,9.0
77,Conditions_Detail_List_Cells = 1,1.0,1.0,2.0,4.0,NaN,NaN,NaN,NaN
79,Conditions_Detail_List_Chemical Actions and Us...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
83,Conditions_Detail_List_Circulatory and Respira...,NaN,NaN,3.0,4.0,2.0,0.0,0.0,6.0
89,Conditions_Detail_List_Diagnosis = 1,30.0,13.0,61.0,39.0,25.0,15.0,39.0,36.0
93,Conditions_Detail_List_Education = 1,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN
97,Conditions_Detail_List_Environment and Public ...,1.0,2.0,10.0,11.0,4.0,4.0,10.0,14.0
104,Conditions_Detail_List_Genetic Phenomena = 1,3.0,0.0,2.0,2.0,NaN,NaN,NaN,NaN
106,Conditions_Detail_List_Health Care Economics a...,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN


# Load Merged Interaction data
- Dropped first categories is done. 
- Only interaction terms are loaded

In [331]:
df1 = pd.read_pickle(r".\df_dummies\df1_dummies.pkl")
df2 = pd.read_pickle(r".\df_dummies\df2_dummies.pkl")
df3 = pd.read_pickle(r".\df_dummies\df3_dummies.pkl")
df4 = pd.read_pickle(r".\df_dummies\df4_dummies.pkl")

## Resample

In [332]:
X_train1, y_train1 = fun_res(df1)
X_train2, y_train2 = fun_res(df2)
X_train3, y_train3 = fun_res(df3)
X_train4, y_train4 = fun_res(df4)

## Create X_y dfs

In [333]:
df1_train = pd.concat([X_train1, y_train1], axis=1)  # X_train y_train have same index
df2_train = pd.concat([X_train2, y_train2], axis=1)  # X_train y_train have same index
df3_train = pd.concat([X_train3, y_train3], axis=1)  # X_train y_train have same index
df4_train = pd.concat([X_train4, y_train4], axis=1)  # X_train y_train have same index

display(X_train1.shape)
display(X_train2.shape)
display(X_train3.shape)
display(X_train4.shape)

(5092, 122)

(8678, 122)

(3854, 122)

(3694, 122)

## Unique cols

In [334]:
dfs = [X_train1, X_train2, X_train3, X_train4]
all_unique_cols = set().union(*(df.columns for df in dfs))
print(f"{len(all_unique_cols)}")

# Columns missing from the dfs
for i, df in enumerate(dfs, start=1):
    missing = all_unique_cols - set(df.columns)
    extra = set(df.columns) - all_unique_cols
    print(f" X_train{i}: shape={df.shape}")
    print(f" Missing cols: {missing if missing else 'None'}")
    print(f" Extra cols:   {extra if extra else 'None'}\n")

122
 X_train1: shape=(5092, 122)
 Missing cols: None
 Extra cols:   None

 X_train2: shape=(8678, 122)
 Missing cols: None
 Extra cols:   None

 X_train3: shape=(3854, 122)
 Missing cols: None
 Extra cols:   None

 X_train4: shape=(3694, 122)
 Missing cols: None
 Extra cols:   None



## Inter_Pivots

In [335]:
inter_cols = [col for col in all_unique_cols if '_x_' in col and 'Enrollment' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, inter_cols)
pivot_merged2 = fun_sparse(2, df2_train, inter_cols)
pivot_merged3 = fun_sparse(3, df3_train, inter_cols)
pivot_merged4 = fun_sparse(4, df4_train, inter_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_inter = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_inter

In [336]:
sparse_inter_train = fun_zeros(pivot_merged_train_inter, 10, True)
display(sparse_inter_train)

Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4
1,BEHAVIORAL_x_Funder_Industry_List = 1,0,1,14,6,3,1,1,1
7,DIETARY_SUPPLEMENT_x_Funder_Industry_List = 1,5,3,23,12,7,6,5,1
11,INTERV_UNSPES_x_Funder_Industry_List = 1,16,27,24,39,20,22,4,5
13,PROCEDURE_x_Funder_Industry_List = 1,9,8,15,12,12,13,4,7
